# 3 · Data skipping and MERGE

Read only the files that matter. Then rewrite only the files that matter.

In [1]:
#@title Setup — run before the session
import os, shutil, subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyspark==3.5.3', 'delta-spark==3.3.2'], check=True)
if not shutil.which('java'):
    subprocess.run('apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless', shell=True, check=True)

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

builder = (SparkSession.builder.master('local[2]')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.sql.shuffle.partitions', '2')
    .config('spark.databricks.delta.properties.defaults.enableDeletionVectors', 'false')
    .config('spark.ui.showConsoleProgress', 'false'))
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('OFF')  # errors still reach Python

PATH = '/tmp/delta_talk/skip_merge'
shutil.rmtree(PATH, ignore_errors=True)
print('Spark', spark.version, 'ready')

Spark 3.5.3 ready


In [2]:
#@title Helpers
import json
from pyspark.sql import functions as F
SCHEMA = 'id INT, gender STRING, age INT, interest STRING'
SPREAD = PATH + '/spread'
ORDERED = PATH + '/ordered'
FILES = PATH + '/three_files'

def files_read(df):
    # Run the query, then ask the executed plan's scan how many files it opened
    df.collect()
    leaves = df._jdf.queryExecution().executedPlan().collectLeaves()
    scans = [leaves.apply(i) for i in range(leaves.size())]
    return sum(s.metrics().get('numFiles').get().value() for s in scans if s.metrics().contains('numFiles'))

def file_labels():
    # Versions 0, 1, 2 of the three-file table each added one file: call them A, B, C
    labels = {}
    if not os.path.exists(FILES): return labels
    for version, name in enumerate('ABC'):
        for line in open(f'{FILES}/_delta_log/{version:020d}.json'):
            action = json.loads(line)
            if 'add' in action: labels[action['add']['path']] = name
    return labels

def short_name(path_col):
    # A, B, C for the three-file table; otherwise the first part of the file name
    labels = F.create_map(*[F.lit(x) for kv in file_labels().items() for x in kv]) if file_labels() else None
    name = F.regexp_extract(path_col, r'([^/]+)$', 1)
    return F.coalesce(labels[name], F.lit('new')) if labels is not None else F.substring(name, 1, 15)

def with_file(df):
    return df.withColumn('file', short_name(F.input_file_name()))

def file_report():
    added, removed = set(), set()
    for f in sorted(os.listdir(FILES + '/_delta_log')):
        if f.endswith('.json'):
            for line in open(f'{FILES}/_delta_log/{f}'):
                action = json.loads(line)
                if 'add' in action: added.add(action['add']['path'])
                if 'remove' in action: removed.add(action['remove']['path'])
    labels = file_labels()
    for f in sorted(added, key=lambda f: labels.get(f, 'z')):
        print(f'{labels.get(f, "new"):4} {"removed by MERGE" if f in removed else "in table"}')

# Part 1 · Data skipping

## 16,000 users, same rows, two layouts, 8 files each

In [3]:
users = spark.range(16000).selectExpr(
    "CAST(id AS INT) AS id",
    "IF(id % 2 = 0, 'Female', 'Male') AS gender",
    "CAST(18 + id % 60 AS INT) AS age",
    "element_at(array('Sports', 'News', 'Social', 'Fashion'), CAST(id % 4 + 1 AS INT)) AS interest")

users.repartition(8).write.format('delta').save(SPREAD)
users.repartitionByRange(8, 'id').write.format('delta').save(ORDERED)

## What does the log know about each file?

In [4]:
def stats(path):
    adds = spark.read.json(f'{path}/_delta_log/*.json').where('add is not null')
    return adds.select(
        short_name('add.path').alias('file'),
        F.get_json_object('add.stats', '$.numRecords').alias('rows'),
        F.get_json_object('add.stats', '$.minValues.id').cast('int').alias('min_id'),
        F.get_json_object('add.stats', '$.maxValues.id').cast('int').alias('max_id')).orderBy('min_id')

stats(SPREAD).show()
stats(ORDERED).show()

+---------------+----+------+------+
|           file|rows|min_id|max_id|
+---------------+----+------+------+
|part-00007-bcd8|2000|     0| 15979|
|part-00002-1f76|2000|     1| 15998|
|part-00003-67c2|2000|     2| 15988|
|part-00004-bfe3|2000|     3| 15990|
|part-00001-f022|2000|     4| 15991|
|part-00000-b120|2000|     5| 15999|
|part-00005-1055|2000|     6| 15995|
|part-00006-51e3|2000|     9| 15997|
+---------------+----+------+------+

+---------------+----+------+------+
|           file|rows|min_id|max_id|
+---------------+----+------+------+
|part-00000-8ba5|1955|     0|  1954|
|part-00001-f695|1856|  1955|  3810|
|part-00002-3002|2034|  3811|  5844|
|part-00003-ecaf|2153|  5845|  7997|
|part-00004-0941|2063|  7998| 10060|
|part-00005-939f|1955| 10061| 12015|
|part-00006-9311|2003| 12016| 14018|
|part-00007-e306|1981| 14019| 15999|
+---------------+----+------+------+



## Which files could contain user 750? How many does Spark actually open?

In [5]:
for name, path in [('spread', SPREAD), ('ordered', ORDERED)]:
    candidates = stats(path).where('min_id <= 750 AND max_id >= 750').count()
    opened = files_read(spark.read.format('delta').load(path).where('id = 750'))
    print(f'{name:8} could contain 750: {candidates} of 8    files opened: {opened}')

spread   could contain 750: 8 of 8    files opened: 8
ordered  could contain 750: 1 of 8    files opened: 1


## And a filter on interest?

In [6]:
for name, path in [('spread', SPREAD), ('ordered', ORDERED)]:
    opened = files_read(spark.read.format('delta').load(path).where("interest = 'News'"))
    print(f'{name:8} files opened: {opened}')

spread   files opened: 8
ordered  files opened: 8


# Part 2 · MERGE

## A table in three files: A, B, C

In [7]:
for rows in [[(100, 'Male', 25, 'Sports'), (150, 'Female', 31, 'News'), (200, 'Male', 35, 'Social')],
             [(75, 'Male', 25, 'Sports'), (125, 'Female', 31, 'News'), (250, 'Male', 35, 'Social'), (300, 'Female', 28, 'Fashion')],
             [(400, 'Male', 25, 'Sports'), (450, 'Female', 31, 'News'), (500, 'Male', 35, 'Social')]]:
    spark.createDataFrame(rows, SCHEMA).coalesce(1).write.format('delta').mode('append').save(FILES)

target = with_file(spark.read.format('delta').load(FILES))
target.groupBy('file').agg(F.min('id').alias('min_id'), F.max('id').alias('max_id'),
                           F.sort_array(F.collect_list('id')).alias('ids')).orderBy('file').show(truncate=False)

+----+------+------+-------------------+
|file|min_id|max_id|ids                |
+----+------+------+-------------------+
|A   |100   |200   |[100, 150, 200]    |
|B   |75    |300   |[75, 125, 250, 300]|
|C   |400   |500   |[400, 450, 500]    |
+----+------+------+-------------------+



## A partner sends a batch: one update, one new user

In [8]:
batch = spark.createDataFrame([(150, 'Female', 31, 'Fashion'), (175, 'Male', 22, 'Social')], SCHEMA)
batch.show()

+---+------+---+--------+
| id|gender|age|interest|
+---+------+---+--------+
|150|Female| 31| Fashion|
|175|  Male| 22|  Social|
+---+------+---+--------+



## Step 1 · Skip, using only the log: which files *could* hold ids 150–175?

In [9]:
stats(FILES).show()
stats(FILES).where('min_id <= 175 AND max_id >= 150').show()

+----+----+------+------+
|file|rows|min_id|max_id|
+----+----+------+------+
|   B|   4|    75|   300|
|   A|   3|   100|   200|
|   C|   3|   400|   500|
+----+----+------+------+

+----+----+------+------+
|file|rows|min_id|max_id|
+----+----+------+------+
|   B|   4|    75|   300|
|   A|   3|   100|   200|
+----+----+------+------+



## Step 2 · Inner join: which files *actually* contain a match?

In [10]:
candidates = target.where("file IN ('A', 'B')")
candidates.join(batch.select('id'), 'id', 'inner').select('id', 'file').show()

+---+----+
| id|file|
+---+----+
|150|   A|
+---+----+



## Step 3 · Full outer join: rebuild file A with the batch

In [11]:
a = target.where("file = 'A'").alias('t')
a.join(batch.alias('s'), 'id', 'full_outer').select(
    'id', F.col('t.interest').alias('old'), F.col('s.interest').alias('new'),
    F.when(F.col('t.file').isNull(), 'INSERT')
     .when(F.col('s.interest').isNull(), 'KEEP')
     .otherwise('UPDATE').alias('action')).orderBy('id').show()

+---+------+-------+------+
| id|   old|    new|action|
+---+------+-------+------+
|100|Sports|   NULL|  KEEP|
|150|  News|Fashion|UPDATE|
|175|  NULL| Social|INSERT|
|200|Social|   NULL|  KEEP|
+---+------+-------+------+



## Now the real MERGE

In [12]:
(DeltaTable.forPath(spark, FILES).alias('t')
    .merge(batch.alias('s'), 't.id = s.id AND t.id BETWEEN 150 AND 175')
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())

spark.read.format('delta').load(FILES).orderBy('id').show()

+---+------+---+--------+
| id|gender|age|interest|
+---+------+---+--------+
| 75|  Male| 25|  Sports|
|100|  Male| 25|  Sports|
|125|Female| 31|    News|
|150|Female| 31| Fashion|
|175|  Male| 22|  Social|
|200|  Male| 35|  Social|
|250|  Male| 35|  Social|
|300|Female| 28| Fashion|
|400|  Male| 25|  Sports|
|450|Female| 31|    News|
|500|  Male| 35|  Social|
+---+------+---+--------+



The same MERGE in SQL:

```sql
MERGE INTO profiles AS t
USING batch AS s
ON t.id = s.id AND t.id BETWEEN 150 AND 175
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
```

## What did MERGE rewrite?

In [13]:
metrics = DeltaTable.forPath(spark, FILES).history(1).first().operationMetrics
for name in ['numSourceRows', 'numTargetRowsUpdated', 'numTargetRowsInserted', 'numTargetRowsCopied',
             'numTargetFilesRemoved', 'numTargetFilesAdded']:
    print(f'{name:24} {metrics[name]}')
print()
file_report()

numSourceRows            2
numTargetRowsUpdated     1
numTargetRowsInserted    1
numTargetRowsCopied      2
numTargetFilesRemoved    1
numTargetFilesAdded      1

A    removed by MERGE
B    in table
C    in table
new  in table


**Downstream, a dashboard counts users by interest. How does it find out what changed?** → next